# Обучение LoRA-адаптера для категории «Легковоспламеняющиеся»

Этот notebook обучает **только `Легковоспламеняющиеся`** на обеих T4.

In [ ]:
# DEPENDENCIES
import subprocess, sys

def pip_install(args):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    print("$", " ".join(cmd))
    subprocess.check_call(cmd)

pip_install([
    "-U",
    "transformers==5.14.0",
    "peft",
    "accelerate",
    "bitsandbytes",
    "safetensors",
    "huggingface_hub",
])

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "fla-core",
        "flash-linear-attention",
        "causal-conv1d",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=False,
)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-U",
    "flash-linear-attention[cuda]",
])

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-U",
    "causal-conv1d",
    "--no-build-isolation",
])

print("=== DEPENDENCIES READY ===")


In [ ]:
# CONFIG
from pathlib import Path
import json
import math
import os
import shlex
import shutil
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm
import torch
from huggingface_hub import snapshot_download
from IPython.display import FileLink, display

ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
FIRE_CATEGORY = "Легковоспламеняющиеся"
MODEL_ID = "Qwen/Qwen3.5-4B"

CONTACT_SHEET_SIZE = 576
PROCESSOR_MAX_SIDE = 448
MAX_IMAGES = 5
JPEG_QUALITY = 88

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

LR = 1e-4
EPOCHS = 1
GRAD_ACCUM_PER_RANK = 4

SAVE_STEPS = 150
SAVE_TOTAL_LIMIT = 3
AMP_INIT_SCALE = 1024.0

ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working/ecup_qwen_ocr_fsdp_FIRE_fulltext")
HF_ROOT = Path("/kaggle/working/hf_cache")

SHEETS_DIR = WORK / "contact_sheets_FIRE"
LOG_DIR = WORK / "logs"
FIRE_DIR = WORK / "qwen35_4b_FIRE_ocr_fsdp_fulltext"

for p in [
    WORK,
    HF_ROOT,
    SHEETS_DIR,
    LOG_DIR,
    FIRE_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_ROOT)
os.environ["HF_HUB_CACHE"] = str(HF_ROOT / "hub")
os.environ["HF_XET_CACHE"] = str(HF_ROOT / "xet")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

assert torch.cuda.is_available()
assert torch.cuda.device_count() == 2, "Select T4 x2"

for i in range(2):
    p = torch.cuda.get_device_properties(i)
    print(
        i,
        torch.cuda.get_device_name(i),
        round(p.total_memory / 2**30, 2),
        "GB",
    )

def disk_status(title):
    total, used, free = shutil.disk_usage("/kaggle/working")
    print(
        f"{title}: used={used/2**30:.2f} GB "
        f"free={free/2**30:.2f} GB"
    )

disk_status("Start")


In [ ]:
# LOCATE DATA / IMAGES / HIGH-QUALITY OCR

def norm_id(x):
    try:
        f = float(x)
        if f.is_integer():
            return str(int(f))
    except Exception:
        pass
    return str(x)

def find_data_csv(root):
    required = {
        "id",
        "name",
        "description",
        "category",
        "label",
    }

    for p in root.rglob("data.csv"):
        try:
            if required <= set(
                pd.read_csv(p, nrows=3).columns
            ):
                return p
        except Exception:
            pass

    raise FileNotFoundError("data.csv not found")

def find_images_root(root):
    known = [
        Path(
            "/kaggle/input/datasets/fabifvue/"
            "ozon2task-images/images"
        ),
        Path(
            "/kaggle/input/ozon2task-images/images"
        ),
        Path(
            "/kaggle/input/ozon2task_images/images"
        ),
    ]

    for p in known:
        if p.exists():
            return p

    candidates = []

    for p in root.rglob("images"):
        if not p.is_dir():
            continue

        dirs = [
            x
            for x in list(p.iterdir())[:100]
            if x.is_dir()
        ]

        if not dirs:
            continue

        score = (
            sum(x.name.isdigit() for x in dirs)
            / len(dirs)
        )

        if score >= 0.7:
            candidates.append(
                (score, p)
            )

    if not candidates:
        raise FileNotFoundError(
            "images/<id> not found"
        )

    return max(
        candidates,
        key=lambda x: x[0],
    )[1]

def find_ocr_products(root):
    exact = list(
        root.rglob(
            "ocr_all_products_high_quality.csv"
        )
    )

    if exact:
        return exact[0]

    for p in root.rglob("*.csv"):
        try:
            cols = set(
                pd.read_csv(
                    p,
                    nrows=2,
                ).columns
            )

            if {
                "id",
                "ocr_text",
                "ocr_char_count",
            } <= cols:
                return p

        except Exception:
            pass

    raise FileNotFoundError(
        "high-quality OCR csv not found"
    )

DATA_CSV = find_data_csv(ROOT)
IMAGES_ROOT = find_images_root(ROOT)
OCR_PRODUCTS_CSV = find_ocr_products(ROOT)

print("DATA  :", DATA_CSV)
print("IMAGES:", IMAGES_ROOT)
print("OCR   :", OCR_PRODUCTS_CSV)

all_df = pd.read_csv(DATA_CSV)

if "Unnamed: 0" in all_df.columns:
    all_df = all_df.drop(
        columns=["Unnamed: 0"]
    )

ocr = pd.read_csv(
    OCR_PRODUCTS_CSV
)

all_df["_id"] = all_df["id"].map(
    norm_id
)

ocr["_id"] = ocr["id"].map(
    norm_id
)

assert all_df["_id"].is_unique
assert ocr["_id"].is_unique

all_df = all_df.merge(
    ocr[["_id", "ocr_text"]],
    on="_id",
    how="left",
    validate="one_to_one",
)

all_df["ocr_text"] = (
    all_df["ocr_text"]
    .fillna("")
    .astype(str)
)

df = (
    all_df[
        all_df["category"]
        == FIRE_CATEGORY
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(df) > 0

print()
print("FIRE rows:", len(df))
print(
    "OCR non-empty:",
    int(
        df["ocr_text"]
        .str.len()
        .gt(0)
        .sum()
    ),
    "/",
    len(df),
)

raw_total = (
    df["name"]
    .fillna("")
    .astype(str)
    .str.len()
    + df["description"]
    .fillna("")
    .astype(str)
    .str.len()
    + df["ocr_text"]
    .fillna("")
    .astype(str)
    .str.len()
)

print(
    "RAW total chars median:",
    int(raw_total.median()),
)
print(
    "RAW total chars p95:",
    int(raw_total.quantile(0.95)),
)
print(
    "RAW total chars p99:",
    int(raw_total.quantile(0.99)),
)
print(
    "RAW total chars max:",
    int(raw_total.max()),
)

display(
    df["label"]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)


In [ ]:
# LINK FIRE IMAGES ONLY
IMG_EXTS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
}

def sort_key(p):
    try:
        return (0, int(p.stem))
    except Exception:
        return (1, p.name)

def get_images(pid):
    folder = (
        IMAGES_ROOT
        / norm_id(pid)
    )

    if not folder.exists():
        return []

    return sorted(
        [
            p
            for p in folder.iterdir()
            if p.is_file()
            and p.suffix.lower()
            in IMG_EXTS
        ],
        key=sort_key,
    )[:MAX_IMAGES]

df["image_paths"] = [
    get_images(x)
    for x in tqdm(
        df["id"],
        desc="Link FIRE images",
    )
]

df["n_images"] = (
    df["image_paths"]
    .map(len)
)

print(
    "Missing image folders:",
    int(
        df["n_images"]
        .eq(0)
        .sum()
    ),
)


In [ ]:
# CONTACT SHEETS FOR FIRE ONLY

def fit_tile(img, box):
    img = img.convert("RGB")
    img.thumbnail(
        box,
        Image.Resampling.LANCZOS,
    )

    canvas = Image.new(
        "RGB",
        box,
        "white",
    )

    canvas.paste(
        img,
        (
            (box[0] - img.width) // 2,
            (box[1] - img.height) // 2,
        ),
    )

    return canvas

def make_sheet(paths, out_path):
    if (
        out_path.exists()
        and out_path.stat().st_size > 0
    ):
        return str(out_path)

    paths = list(paths)[:MAX_IMAGES]

    if not paths:
        Image.new(
            "RGB",
            (
                CONTACT_SHEET_SIZE,
                CONTACT_SHEET_SIZE,
            ),
            "white",
        ).save(
            out_path,
            "JPEG",
            quality=JPEG_QUALITY,
        )
        return str(out_path)

    n = len(paths)

    if n == 1:
        cols, rows = 1, 1
    elif n <= 4:
        cols, rows = 2, math.ceil(n / 2)
    else:
        cols, rows = 3, 2

    gap = 4

    tw = (
        CONTACT_SHEET_SIZE
        - gap * (cols - 1)
    ) // cols

    th = (
        CONTACT_SHEET_SIZE
        - gap * (rows - 1)
    ) // rows

    sheet = Image.new(
        "RGB",
        (
            CONTACT_SHEET_SIZE,
            CONTACT_SHEET_SIZE,
        ),
        "white",
    )

    for i, p in enumerate(paths):
        try:
            with Image.open(p) as im:
                tile = fit_tile(
                    im,
                    (tw, th),
                )
        except Exception:
            tile = Image.new(
                "RGB",
                (tw, th),
                "white",
            )

        sheet.paste(
            tile,
            (
                (i % cols) * (tw + gap),
                (i // cols) * (th + gap),
            ),
        )

    sheet.save(
        out_path,
        "JPEG",
        quality=JPEG_QUALITY,
        optimize=False,
    )

    return str(out_path)

def sheet_worker(item):
    i, pid, paths = item
    out_path = (
        SHEETS_DIR
        / f"{norm_id(pid)}.jpg"
    )
    return (
        i,
        make_sheet(
            paths,
            out_path,
        ),
    )

items = [
    (
        i,
        row["id"],
        row["image_paths"],
    )
    for i, row in df.iterrows()
]

sheet_paths = [None] * len(df)

with ThreadPoolExecutor(
    max_workers=16
) as ex:
    futures = [
        ex.submit(
            sheet_worker,
            item,
        )
        for item in items
    ]

    for f in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="FIRE contact sheets",
    ):
        i, p = f.result()
        sheet_paths[i] = p

df["sheet_path"] = sheet_paths

MANIFEST = (
    WORK
    / "train_FIRE_fulltext_manifest.csv"
)

df[
    [
        "id",
        "name",
        "description",
        "category",
        "label",
        "sheet_path",
        "ocr_text",
        "n_images",
    ]
].to_csv(
    MANIFEST,
    index=False,
)

print("MANIFEST:", MANIFEST)
disk_status("After FIRE sheets")


In [ ]:
# DOWNLOAD QWEN 

MODEL_PATH = snapshot_download(
    repo_id=MODEL_ID,
    cache_dir=str(
        HF_ROOT / "hub"
    ),
    max_workers=4,
)

xet = HF_ROOT / "xet"

if xet.exists():
    shutil.rmtree(
        xet,
        ignore_errors=True,
    )
    xet.mkdir(
        parents=True,
        exist_ok=True,
    )

print("MODEL_PATH:", MODEL_PATH)
disk_status("After model")


In [ ]:
# WRITE FINAL FIRE TRAINER + FSDP CONFIGS

TRAIN_SCRIPT = (
    WORK
    / "train_FIRE_qwen_fsdp_fulltext.py"
)

TRAINER_CODE = '\nimport argparse\nimport json\nimport os\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport torch\nfrom torch.utils.data import Dataset\n\nfrom transformers import (\n    AutoProcessor,\n    BitsAndBytesConfig,\n    Qwen3_5ForConditionalGeneration,\n    Trainer,\n    TrainerCallback,\n    TrainingArguments,\n    set_seed,\n)\nfrom transformers.trainer_utils import get_last_checkpoint\n\nfrom peft import (\n    LoraConfig,\n    get_peft_model,\n    prepare_model_for_kbit_training,\n)\n\ntry:\n    import peft.tuners.lora.torchao as peft_torchao\n    peft_torchao.is_torchao_available = lambda: False\nexcept Exception:\n    pass\n\n\nFIRE_CATEGORY = "Легковоспламеняющиеся"\n\nFIRE_RULES = """Правила Легковоспламеняющиеся:\n- относится: самостоятельный источник воспламенения; содержит горючее вещество/ЛВЖ/горючий газ; опасный товар входит в комплект;\n- не относится: устройство лишь используется с огнем/топливом, но не содержит его;\n- не относится: горючее содержимое отсутствует в поставке;\n- не относится: источник воспламенения встроен;\n- не относится: горючий материал только компонент;\n- не относится: опасный предмет не входит в комплект."""\n\n\ndef parse_args():\n    p = argparse.ArgumentParser()\n    p.add_argument("--model_path", required=True)\n    p.add_argument("--manifest", required=True)\n    p.add_argument("--output_dir", required=True)\n    p.add_argument("--r", type=int, default=16)\n    p.add_argument("--alpha", type=int, default=32)\n    p.add_argument("--dropout", type=float, default=0.05)\n    p.add_argument("--lr", type=float, default=1e-4)\n    p.add_argument("--epochs", type=float, default=1.0)\n    p.add_argument("--grad_accum", type=int, default=4)\n    p.add_argument("--weight_decay", type=float, default=0.01)\n    p.add_argument("--warmup_ratio", type=float, default=0.05)\n    p.add_argument("--seed", type=int, default=42)\n    p.add_argument("--visual_side", type=int, default=448)\n    p.add_argument("--save_steps", type=int, default=150)\n    p.add_argument("--save_total_limit", type=int, default=3)\n    p.add_argument("--amp_init_scale", type=float, default=1024.0)\n    p.add_argument("--smoke", action="store_true")\n    return p.parse_args()\n\n\ndef raw_text(x):\n    if x is None:\n        return ""\n    try:\n        if pd.isna(x):\n            return ""\n    except Exception:\n        pass\n    return str(x)\n\n\ndef build_prompt(row):\n    name = raw_text(row.get("name", ""))\n    desc = raw_text(row.get("description", ""))\n    ocr = raw_text(row.get("ocr_text", ""))\n\n    ocr_block = ""\n    if ocr.strip():\n        ocr_block = (\n            "\\n\\nТекст, автоматически распознанный на фотографиях товара (OCR).\\n"\n            "OCR может содержать ошибки и склеенные слова. Используй его как "\n            "дополнительный источник и сопоставляй с названием, описанием и изображением.\\n\\n"\n            f"{ocr}\\n"\n        )\n\n    return f"""Ты решаешь бинарную классификацию товара.\n\n{FIRE_RULES}\n\nНазвание:\n{name}\n\nОписание:\n{desc}{ocr_block}\n\nНа изображении объединены все фотографии товара.\n\nПредскажи целевую метку из обучающей разметки.\nОтветь строго одним символом: 0 или 1.\n\nОтвет:"""\n\n\ndef balanced_full_coverage(df, seed):\n    rng = np.random.default_rng(seed)\n    labels = df["label"].astype(int).to_numpy()\n    classes = sorted(np.unique(labels).tolist())\n    by_class = {c: np.flatnonzero(labels == c) for c in classes}\n    target_n = max(len(v) for v in by_class.values())\n\n    parts = []\n    for c in classes:\n        idx = by_class[c].copy()\n        rng.shuffle(idx)\n        parts.append(idx)\n\n        extra_n = target_n - len(idx)\n        if extra_n > 0:\n            parts.append(rng.choice(idx, size=extra_n, replace=True))\n\n    out = np.concatenate(parts)\n    rng.shuffle(out)\n    return df.iloc[out].reset_index(drop=True)\n\n\nclass ProductDataset(Dataset):\n    def __init__(self, frame):\n        self.records = frame.to_dict("records")\n\n    def __len__(self):\n        return len(self.records)\n\n    def __getitem__(self, idx):\n        return self.records[idx]\n\n\nclass ProductCollator:\n    def __init__(self, processor, zero_id, one_id):\n        self.processor = processor\n        self.zero_id = zero_id\n        self.one_id = one_id\n\n    def __call__(self, features):\n        if len(features) != 1:\n            raise RuntimeError(f"Expected per-device batch=1, got {len(features)}")\n\n        row = features[0]\n        target_id = self.one_id if int(row["label"]) == 1 else self.zero_id\n\n        messages = [\n            {\n                "role": "user",\n                "content": [\n                    {"type": "image"},\n                    {"type": "text", "text": build_prompt(row)},\n                ],\n            }\n        ]\n\n        try:\n            text = self.processor.apply_chat_template(\n                messages,\n                tokenize=False,\n                add_generation_prompt=True,\n                enable_thinking=False,\n            )\n        except TypeError:\n            text = self.processor.apply_chat_template(\n                messages,\n                tokenize=False,\n                add_generation_prompt=True,\n            )\n\n        with Image.open(row["sheet_path"]) as im:\n            image = im.convert("RGB").copy()\n\n        batch = self.processor(\n            text=[text],\n            images=[image],\n            padding=False,\n            return_tensors="pt",\n        )\n\n        batch["target_label"] = torch.tensor(\n            [target_id],\n            dtype=torch.long,\n        )\n\n        return batch\n\n\nclass LastTokenTrainer(Trainer):\n    def compute_loss(\n        self,\n        model,\n        inputs,\n        return_outputs=False,\n        num_items_in_batch=None,\n    ):\n        inputs = dict(inputs)\n        target = inputs.pop("target_label")\n\n        outputs = model(\n            **inputs,\n            use_cache=False,\n            logits_to_keep=1,\n        )\n\n        logits = outputs.logits[:, -1, :].float()\n        target = target.to(device=logits.device, dtype=torch.long)\n\n        loss = torch.nn.functional.cross_entropy(\n            logits,\n            target,\n        )\n\n        if not hasattr(self, "_ecup_loss_debug_done"):\n            self._ecup_loss_debug_done = True\n            rank = int(os.environ.get("RANK", "0"))\n            print(\n                f"RANK={rank} LAST-TOKEN LOSS ACTIVE | "\n                f"logits={tuple(logits.shape)} | target={target.tolist()}",\n                flush=True,\n            )\n\n        if return_outputs:\n            return loss, outputs\n\n        return loss\n\n\nclass CheckpointPrintCallback(TrainerCallback):\n    def __init__(self, output_dir):\n        self.output_dir = str(output_dir)\n\n    def on_save(self, args, state, control, **kwargs):\n        if state.is_local_process_zero:\n            path = Path(self.output_dir) / f"checkpoint-{state.global_step}"\n            print(\n                f"\\n=== CHECKPOINT SAVED: {path} ===\\n",\n                flush=True,\n            )\n        return control\n\n\ndef find_text_linear_modules(model):\n    targets = []\n\n    for name, module in model.named_modules():\n        if not name.startswith("model.language_model."):\n            continue\n        if "linear" not in module.__class__.__name__.lower():\n            continue\n        if name.endswith("lm_head"):\n            continue\n        targets.append(name)\n\n    targets = sorted(set(targets))\n\n    if len(targets) < 50:\n        raise RuntimeError(\n            f"Too few text-linear LoRA targets: {len(targets)}"\n        )\n\n    return targets\n\n\ndef prepare_processor(model_path, visual_side):\n    processor = AutoProcessor.from_pretrained(\n        model_path,\n        local_files_only=True,\n    )\n\n    processor.tokenizer.padding_side = "left"\n\n    if processor.tokenizer.pad_token_id is None:\n        processor.tokenizer.pad_token = processor.tokenizer.eos_token\n\n    processor.image_processor.size["longest_edge"] = visual_side * visual_side\n    processor.image_processor.size["shortest_edge"] = min(\n        224 * 224,\n        visual_side * visual_side,\n    )\n\n    zero = processor.tokenizer.encode(\n        "0",\n        add_special_tokens=False,\n    )\n    one = processor.tokenizer.encode(\n        "1",\n        add_special_tokens=False,\n    )\n\n    if len(zero) != 1 or len(one) != 1:\n        raise RuntimeError(\n            f"0/1 must each be one token: 0={zero}, 1={one}"\n        )\n\n    return processor, zero[0], one[0]\n\n\ndef load_model(args):\n    qcfg = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_quant_type="nf4",\n        bnb_4bit_use_double_quant=True,\n        bnb_4bit_compute_dtype=torch.float16,\n        bnb_4bit_quant_storage=torch.float32,\n    )\n\n    model = Qwen3_5ForConditionalGeneration.from_pretrained(\n        args.model_path,\n        quantization_config=qcfg,\n        dtype=torch.float32,\n        attn_implementation="sdpa",\n        local_files_only=True,\n    )\n\n    model.tie_weights()\n    model.config.use_cache = False\n\n    # FSDP + Qwen3.5 vision fix discovered in the working BAD notebook.\n    visual_cls = type(model.model.visual)\n    visual_cls.dtype = property(lambda self: torch.float16)\n\n    print(\n        f"VISION DTYPE PATCH: {model.model.visual.dtype}",\n        flush=True,\n    )\n\n    model = prepare_model_for_kbit_training(\n        model,\n        use_gradient_checkpointing=True,\n        gradient_checkpointing_kwargs={"use_reentrant": False},\n    )\n\n    try:\n        model.enable_input_require_grads()\n    except Exception:\n        pass\n\n    targets = find_text_linear_modules(model)\n\n    lcfg = LoraConfig(\n        r=args.r,\n        lora_alpha=args.alpha,\n        lora_dropout=args.dropout,\n        bias="none",\n        target_modules=targets,\n        task_type="CAUSAL_LM",\n    )\n\n    model = get_peft_model(\n        model,\n        lcfg,\n        autocast_adapter_dtype=False,\n    )\n\n    return model, targets\n\n\ndef make_training_frame(df, seed, smoke):\n    part = df[df["category"] == FIRE_CATEGORY].copy().reset_index(drop=True)\n\n    if len(part) == 0:\n        raise RuntimeError(f"No rows for category {FIRE_CATEGORY!r}")\n\n    if smoke:\n        part["_raw_chars"] = (\n            part["name"].fillna("").astype(str).str.len()\n            + part["description"].fillna("").astype(str).str.len()\n            + part["ocr_text"].fillna("").astype(str).str.len()\n        )\n\n        # Enough long real samples for a 3-step, 2-rank smoke.\n        part = (\n            part.sort_values("_raw_chars", ascending=False)\n            .head(8)\n            .drop(columns=["_raw_chars"])\n            .reset_index(drop=True)\n        )\n\n        print(\n            "SMOKE RAW CHARS:",\n            [\n                len(raw_text(r["name"]))\n                + len(raw_text(r["description"]))\n                + len(raw_text(r["ocr_text"]))\n                for _, r in part.iterrows()\n            ],\n            flush=True,\n        )\n\n        return part\n\n    return balanced_full_coverage(part, seed)\n\n\ndef main():\n    args = parse_args()\n\n    set_seed(args.seed)\n    random.seed(args.seed)\n    np.random.seed(args.seed)\n\n    rank = int(os.environ.get("RANK", "0"))\n    local_rank = int(os.environ.get("LOCAL_RANK", "0"))\n    world = int(os.environ.get("WORLD_SIZE", "1"))\n\n    torch.cuda.set_device(local_rank)\n\n    print(\n        f"CUDA BIND: rank={rank} local_rank={local_rank} "\n        f"device=cuda:{torch.cuda.current_device()}",\n        flush=True,\n    )\n    print(\n        f"RANK={rank} WORLD_SIZE={world} CATEGORY={FIRE_CATEGORY}",\n        flush=True,\n    )\n\n    if world != 2:\n        raise RuntimeError(\n            f"Expected exactly 2 distributed processes, got {world}"\n        )\n\n    df = pd.read_csv(args.manifest)\n    df["ocr_text"] = df["ocr_text"].fillna("").astype(str)\n\n    train_df = make_training_frame(\n        df,\n        args.seed,\n        args.smoke,\n    )\n\n    if rank == 0:\n        category_rows = int(\n            (df["category"] == FIRE_CATEGORY).sum()\n        )\n\n        raw_chars = (\n            train_df["name"].fillna("").astype(str).str.len()\n            + train_df["description"].fillna("").astype(str).str.len()\n            + train_df["ocr_text"].fillna("").astype(str).str.len()\n        )\n\n        print("UNIQUE CATEGORY ROWS:", category_rows, flush=True)\n        print("EFFECTIVE TRAIN ROWS:", len(train_df), flush=True)\n        print("RAW TEXT median chars:", int(raw_chars.median()), flush=True)\n        print("RAW TEXT max chars:", int(raw_chars.max()), flush=True)\n        print("NO DESCRIPTION/OCR CHAR TRUNCATION", flush=True)\n\n    processor, zero_id, one_id = prepare_processor(\n        args.model_path,\n        args.visual_side,\n    )\n\n    model, targets = load_model(args)\n\n    model_cuda_devices = sorted(\n        {\n            str(p.device)\n            for p in model.parameters()\n            if p.device.type == "cuda"\n        }\n    )\n\n    print(\n        f"RANK={rank} MODEL CUDA DEVICES BEFORE FSDP:",\n        model_cuda_devices,\n        flush=True,\n    )\n\n    if rank == 0:\n        model.print_trainable_parameters()\n        print(\n            "LoRA target count:",\n            len(targets),\n            flush=True,\n        )\n\n    dataset = ProductDataset(train_df)\n    collator = ProductCollator(\n        processor,\n        zero_id,\n        one_id,\n    )\n\n    out_dir = Path(args.output_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    train_args = TrainingArguments(\n        output_dir=str(out_dir),\n        per_device_train_batch_size=1,\n        gradient_accumulation_steps=1 if args.smoke else args.grad_accum,\n        learning_rate=args.lr,\n        weight_decay=args.weight_decay,\n        num_train_epochs=1 if args.smoke else args.epochs,\n        max_steps=3 if args.smoke else -1,\n        warmup_ratio=0.0 if args.smoke else args.warmup_ratio,\n        lr_scheduler_type="cosine",\n        fp16=True,\n        bf16=False,\n        logging_strategy="steps",\n        logging_steps=1,\n        logging_first_step=True,\n        save_strategy="no" if args.smoke else "steps",\n        save_steps=args.save_steps,\n        save_total_limit=args.save_total_limit,\n        save_only_model=False,\n        report_to="none",\n        remove_unused_columns=False,\n        dataloader_num_workers=0,\n        dataloader_pin_memory=True,\n        optim="adamw_torch",\n        seed=args.seed,\n        data_seed=args.seed,\n        gradient_checkpointing=False,\n        max_grad_norm=1.0,\n        disable_tqdm=False,\n    )\n\n    trainer = LastTokenTrainer(\n        model=model,\n        args=train_args,\n        train_dataset=dataset,\n        data_collator=collator,\n        callbacks=[\n            CheckpointPrintCallback(out_dir),\n        ],\n    )\n\n    fsdp_plugin = getattr(\n        trainer.accelerator.state,\n        "fsdp_plugin",\n        None,\n    )\n\n    if fsdp_plugin is not None:\n        from peft.utils.other import fsdp_auto_wrap_policy\n\n        fsdp_plugin.auto_wrap_policy = fsdp_auto_wrap_policy(\n            trainer.model\n        )\n\n        print(\n            f"RANK={rank} PEFT FSDP AUTO WRAP POLICY: READY",\n            flush=True,\n        )\n\n    amp_scaler = getattr(\n        trainer.accelerator,\n        "scaler",\n        None,\n    )\n\n    if amp_scaler is not None:\n        if getattr(amp_scaler, "_scale", None) is None:\n            amp_scaler._init_scale = float(args.amp_init_scale)\n\n        print(\n            f"RANK={rank} AMP INIT SCALE: "\n            f"{amp_scaler._init_scale}",\n            flush=True,\n        )\n\n    resume_checkpoint = None\n\n    if not args.smoke:\n        resume_checkpoint = get_last_checkpoint(\n            str(out_dir)\n        )\n\n    if rank == 0:\n        if resume_checkpoint is None:\n            print(\n                "AUTO RESUME: no checkpoint -> starting normally",\n                flush=True,\n            )\n        else:\n            print(\n                f"AUTO RESUME: {resume_checkpoint}",\n                flush=True,\n            )\n\n    result = trainer.train(\n        resume_from_checkpoint=resume_checkpoint,\n    )\n\n    amp_scaler = getattr(\n        trainer.accelerator,\n        "scaler",\n        None,\n    )\n\n    if amp_scaler is not None:\n        print(\n            f"RANK={rank} AMP FINAL SCALE: "\n            f"{amp_scaler.get_scale()}",\n            flush=True,\n        )\n        print(\n            f"RANK={rank} LAST OPTIMIZER STEP SKIPPED: "\n            f"{trainer.accelerator.optimizer_step_was_skipped}",\n            flush=True,\n        )\n\n    if args.smoke:\n        if rank == 0:\n            print(\n                "=== 2xT4 FIRE FSDP FULL-TEXT SMOKE PASSED ===",\n                flush=True,\n            )\n        return\n\n    # Training checkpoints are sharded for efficient FSDP resume.\n    # Final export is switched to a full state dict so the adapter\n    # can be loaded normally outside the training job.\n    if trainer.is_fsdp_enabled:\n        trainer.accelerator.state.fsdp_plugin.set_state_dict_type(\n            "FULL_STATE_DICT"\n        )\n\n    trainer.accelerator.wait_for_everyone()\n\n    state_dict = trainer.accelerator.get_state_dict(\n        trainer.model\n    )\n\n    if trainer.args.should_save:\n        unwrapped = trainer.accelerator.unwrap_model(\n            trainer.model\n        )\n\n        unwrapped.save_pretrained(\n            out_dir,\n            state_dict=state_dict,\n            safe_serialization=True,\n        )\n\n        summary = {\n            "base_model": "Qwen/Qwen3.5-4B",\n            "category": FIRE_CATEGORY,\n            "training_scope": "full labeled FIRE data, no holdout",\n            "input": (\n                "full name + full description + "\n                "448x448 visual contact sheet + full high-quality OCR"\n            ),\n            "text_truncation": "none",\n            "visual_side": args.visual_side,\n            "unique_rows": int(\n                (df["category"] == FIRE_CATEGORY).sum()\n            ),\n            "effective_rows_after_balancing": int(\n                len(train_df)\n            ),\n            "global_batch_size": int(\n                2 * args.grad_accum\n            ),\n            "epochs": args.epochs,\n            "lr": args.lr,\n            "lora_r": args.r,\n            "lora_alpha": args.alpha,\n            "lora_dropout": args.dropout,\n            "lora_target_count": len(targets),\n            "checkpoint_every_optimizer_steps": args.save_steps,\n            "checkpoint_keep_last": args.save_total_limit,\n            "amp_init_scale": args.amp_init_scale,\n            "train_metrics": result.metrics,\n        }\n\n        (out_dir / "training_summary.json").write_text(\n            json.dumps(\n                summary,\n                ensure_ascii=False,\n                indent=2,\n            ),\n            encoding="utf-8",\n        )\n\n        (out_dir / "target_modules.json").write_text(\n            json.dumps(\n                targets,\n                ensure_ascii=False,\n                indent=2,\n            ),\n            encoding="utf-8",\n        )\n\n        print(\n            "=== FIRE ADAPTER SAVED ===",\n            out_dir,\n            flush=True,\n        )\n        print(\n            json.dumps(\n                summary,\n                ensure_ascii=False,\n                indent=2,\n            ),\n            flush=True,\n        )\n\n    trainer.accelerator.wait_for_everyone()\n\n\nif __name__ == "__main__":\n    main()\n'

TRAIN_SCRIPT.write_text(
    TRAINER_CODE,
    encoding="utf-8",
)

compile(
    TRAINER_CODE,
    str(TRAIN_SCRIPT),
    "exec",
)

FSDP_FAST = (
    WORK
    / "fsdp_FIRE_2xt4.yaml"
)

FSDP_OFFLOAD = (
    WORK
    / "fsdp_FIRE_2xt4_offload.yaml"
)

def fsdp_yaml(offload):
    return f'''compute_environment: LOCAL_MACHINE
debug: false
distributed_type: FSDP
downcast_bf16: 'no'
fsdp_config:
  fsdp_auto_wrap_policy: TRANSFORMER_BASED_WRAP
  fsdp_backward_prefetch_policy: BACKWARD_PRE
  fsdp_cpu_ram_efficient_loading: false
  fsdp_forward_prefetch: false
  fsdp_offload_params: {str(offload).lower()}
  fsdp_sharding_strategy: FULL_SHARD
  fsdp_state_dict_type: SHARDED_STATE_DICT
  fsdp_sync_module_states: true
  fsdp_transformer_layer_cls_to_wrap: Qwen3_5DecoderLayer
  fsdp_use_orig_params: false
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: true
use_cpu: false
'''

FSDP_FAST.write_text(
    fsdp_yaml(False),
    encoding="utf-8",
)

FSDP_OFFLOAD.write_text(
    fsdp_yaml(True),
    encoding="utf-8",
)

print("Trainer:", TRAIN_SCRIPT)
print("Fast FSDP:", FSDP_FAST)
print("Offload fallback:", FSDP_OFFLOAD)
print("Syntax: OK")


In [ ]:
# LIVE DISTRIBUTED RUNNER

def run_distributed(
    output_dir,
    config_path,
    smoke=False,
    log_name="run.log",
):
    log_path = (
        LOG_DIR / log_name
    )

    cmd = [
        sys.executable,
        "-m",
        "accelerate.commands.launch",
        "--config_file",
        str(config_path),
        str(TRAIN_SCRIPT),
        "--model_path",
        str(MODEL_PATH),
        "--manifest",
        str(MANIFEST),
        "--output_dir",
        str(output_dir),
        "--r",
        str(LORA_R),
        "--alpha",
        str(LORA_ALPHA),
        "--dropout",
        str(LORA_DROPOUT),
        "--lr",
        str(LR),
        "--epochs",
        str(EPOCHS),
        "--grad_accum",
        str(GRAD_ACCUM_PER_RANK),
        "--visual_side",
        str(PROCESSOR_MAX_SIDE),
        "--save_steps",
        str(SAVE_STEPS),
        "--save_total_limit",
        str(SAVE_TOTAL_LIMIT),
        "--amp_init_scale",
        str(AMP_INIT_SCALE),
        "--seed",
        str(SEED),
    ]

    if smoke:
        cmd.append("--smoke")

    env = os.environ.copy()
    env["TOKENIZERS_PARALLELISM"] = "false"
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTORCH_ALLOC_CONF"] = (
        "expandable_segments:True"
    )

    command = " ".join(
        shlex.quote(str(x))
        for x in cmd
    )

    shell_command = (
        "set -o pipefail; "
        + command
        + " 2>&1 | tee "
        + shlex.quote(
            str(log_path)
        )
    )

    print()
    print("=" * 100)
    print("LIVE DISTRIBUTED FIRE TRAINING")
    print("=" * 100)
    print("$", command)
    print()
    sys.stdout.flush()

    p = subprocess.run(
        [
            "/bin/bash",
            "-lc",
            shell_command,
        ],
        env=env,
    )

    log = ""

    if log_path.exists():
        log = (
            log_path
            .read_text(
                encoding="utf-8",
                errors="ignore",
            )
        )

    return (
        p.returncode,
        log,
        log_path,
    )


## FIRE smoke - обе T4

Smoke берёт самые длинные реальные FIRE-карточки и делает 3 optimizer steps.

Проверяем уже исправленную конфигурацию:
- две T4 действительно разделены по rank;
- FSDP работает;
- last-token loss активен;
- grad norm конечный;
- optimizer step не skipped;
- полный description/OCR + 448×448 помещаются.

Сначала `FULL_SHARD`; CPU offload используется только если будет OOM.

In [ ]:
# SELECT FIRE FSDP MODE

SMOKE_DIR = (
    WORK
    / "smoke_FIRE_fulltext"
)

shutil.rmtree(
    SMOKE_DIR,
    ignore_errors=True,
)

rc, log, _ = run_distributed(
    SMOKE_DIR,
    FSDP_FAST,
    smoke=True,
    log_name="smoke_FIRE_fsdp_fast.log",
)

if rc == 0:
    SELECTED_FSDP_CONFIG = FSDP_FAST
    SELECTED_FSDP_MODE = "FULL_SHARD"

elif "out of memory" in log.lower():
    print(
        "\nFAST FSDP OOM -> "
        "CPU parameter offload, "
        "still FULL TEXT"
    )

    shutil.rmtree(
        SMOKE_DIR,
        ignore_errors=True,
    )

    rc, log, _ = run_distributed(
        SMOKE_DIR,
        FSDP_OFFLOAD,
        smoke=True,
        log_name=(
            "smoke_FIRE_fsdp_offload.log"
        ),
    )

    assert rc == 0, (
        "FIRE FSDP full-text smoke "
        "with CPU offload also failed"
    )

    SELECTED_FSDP_CONFIG = (
        FSDP_OFFLOAD
    )

    SELECTED_FSDP_MODE = (
        "FULL_SHARD + CPU PARAM OFFLOAD"
    )

else:
    raise RuntimeError(
        "FIRE FSDP smoke failed "
        "for a non-OOM reason"
    )

shutil.rmtree(
    SMOKE_DIR,
    ignore_errors=True,
)

print()
print("=== FIRE CONFIG SELECTED ===")
print("Mode:", SELECTED_FSDP_MODE)
print("Description/OCR truncation: NONE")
print("Visual side:", PROCESSOR_MAX_SIDE)
print("Checkpoint every:", SAVE_STEPS, "steps")
print("Loss logging: EVERY optimizer step")


## TRAIN FIRE — обе T4

- каждые 150 optimizer steps сохраняется checkpoint;
- если сессия обучения оборвалась, повторный запуск этой же ячейки автоматически найдёт последний `checkpoint-*` и продолжит обучение;
- `loss`, `grad_norm`, `learning_rate`, `epoch` печатаются каждый optimizer step прямо под ячейкой.

In [ ]:
# TRAIN FIRE — RESUMABLE

FIRE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

existing_checkpoints = sorted(
    FIRE_DIR.glob("checkpoint-*"),
    key=lambda p: (
        int(
            p.name.split("-")[-1]
        )
        if p.name.split("-")[-1].isdigit()
        else -1
    ),
)

if existing_checkpoints:
    print(
        "Existing checkpoints:",
        [
            p.name
            for p in existing_checkpoints
        ],
    )
    print(
        "Trainer will AUTO-RESUME "
        "from the latest one."
    )
else:
    print(
        "No FIRE checkpoints yet. "
        "Starting from zero."
    )

rc, log, FIRE_LOG = run_distributed(
    FIRE_DIR,
    SELECTED_FSDP_CONFIG,
    smoke=False,
    log_name="train_FIRE_fsdp.log",
)

assert rc == 0, (
    f"FIRE FSDP training failed. "
    f"Log: {FIRE_LOG}"
)

assert (
    FIRE_DIR
    / "adapter_config.json"
).exists()

assert list(
    FIRE_DIR.glob(
        "adapter_model*.safetensors"
    )
)

print()
print("=== FIRE ADAPTER READY ===")
print("Directory:", FIRE_DIR)


In [ ]:
# CHECKPOINT / ADAPTER STATUS

checkpoints = sorted(
    FIRE_DIR.glob("checkpoint-*"),
    key=lambda p: (
        int(
            p.name.split("-")[-1]
        )
        if p.name.split("-")[-1].isdigit()
        else -1
    ),
)

print(
    "Saved checkpoints:",
    [p.name for p in checkpoints],
)

for p in sorted(
    FIRE_DIR.iterdir()
):
    if p.is_file():
        print(
            p.name,
            f"{p.stat().st_size/2**20:.2f} MB",
        )


In [ ]:
# PACKAGE FINAL FIRE ADAPTER ONLY

EXPORT_DIR = (
    Path("/kaggle/working")
    / "qwen35_FIRE_ocr_fsdp_fulltext_export"
)

shutil.rmtree(
    EXPORT_DIR,
    ignore_errors=True,
)

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

patterns = [
    "adapter_config.json",
    "adapter_model*.safetensors",
    "training_summary.json",
    "target_modules.json",
]

copied = []

for pattern in patterns:
    for src in FIRE_DIR.glob(pattern):
        dst = EXPORT_DIR / src.name
        shutil.copy2(
            src,
            dst,
        )
        copied.append(dst)

assert (
    EXPORT_DIR
    / "adapter_config.json"
).exists()

assert list(
    EXPORT_DIR.glob(
        "adapter_model*.safetensors"
    )
)

metadata = {
    "base_model": MODEL_ID,
    "category": FIRE_CATEGORY,
    "training": (
        "FSDP-QLoRA on both T4s"
    ),
    "fsdp_mode": SELECTED_FSDP_MODE,
    "description_truncation": None,
    "ocr_truncation": None,
    "visual_side": PROCESSOR_MAX_SIDE,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "lr": LR,
    "epochs": EPOCHS,
    "global_batch_size": (
        2 * GRAD_ACCUM_PER_RANK
    ),
    "checkpoint_every_steps": SAVE_STEPS,
}

(
    EXPORT_DIR
    / "export_metadata.json"
).write_text(
    json.dumps(
        metadata,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

ZIP = Path(
    shutil.make_archive(
        "/kaggle/working/"
        "qwen35_FIRE_ocr_fsdp_fulltext_adapter",
        "zip",
        root_dir=str(EXPORT_DIR),
    )
)

print()
print("FINAL FIRE ZIP:", ZIP)
print(
    "Size:",
    round(
        ZIP.stat().st_size
        / 2**20,
        2,
    ),
    "MB",
)

display(
    FileLink(
        str(ZIP)
    )
)
